# CV Assignment 2 — Problem Statement 3
## Advanced Object Tracking and Detection in Video Streams

**Course:** Computer Vision (S2-25_AIMLCZG525)
**Group:** _<fill in group name/number>_
**File name on submission:** `CV_assignment2_<group>_3` (rename before export, per assignment instructions)

**Approach:** Fine-tune a Faster R-CNN detector, then track detected objects across frames
with a Kalman-filter-based SORT tracker that includes (a) a temporal consistency check
(an identity is only confirmed after several consecutive matched frames) and (b) an
adaptive IoU gate that loosens matching tolerance for fast-moving objects.

**Data note:** This notebook defaults to a **synthetic multi-object sequence** (bouncing
coloured rectangles with known ground truth) so every cell below is runnable end-to-end
without first obtaining SportsMOT access. To switch to real data, download a SportsMOT
sequence in MOTChallenge format (`img1/*.jpg` + `gt/gt.txt`) into `DATA_ROOT` below and set
`USE_SYNTHETIC = False`. Nothing else needs to change — the rest of the pipeline reads
from the same `frames` / `gt_df` variables either way.


## 0. Setup

Runtime > Change runtime type > select a GPU (e.g. T4) before running this cell.


In [ ]:
!pip install -q motmetrics filterpy torchmetrics


In [ ]:
import os
import glob
import time
import random

import numpy as np

# motmetrics (as of its latest release on PyPI) still calls np.asfarray, which
# NumPy 2.x removed — shim it back in rather than downgrading NumPy (a NumPy
# downgrade risks breaking the torch/opencv binaries Colab ships).
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

import pandas as pd
import cv2

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

import motmetrics as mm
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Data Preprocessing

### 1.1 Load a sequence (synthetic fallback or real SportsMOT)


In [ ]:
class SyntheticMOTSequence:
    """Generates a synthetic multi-object video sequence with ground-truth boxes/IDs,
    used as a runnable stand-in for a SportsMOT clip."""

    def __init__(self, n_frames=150, n_objects=6, width=640, height=384, seed=0):
        self.n_frames = n_frames
        self.width = width
        self.height = height
        rng = np.random.RandomState(seed)
        self.objects = []
        for i in range(n_objects):
            w, h = int(rng.randint(30, 50)), int(rng.randint(60, 90))
            x = float(rng.randint(0, width - w))
            y = float(rng.randint(0, height - h))
            vx = float(rng.uniform(-6, 6))
            vy = float(rng.uniform(-6, 6))
            vx = vx if abs(vx) > 1 else 2.0
            vy = vy if abs(vy) > 1 else 2.0
            color = tuple(int(c) for c in rng.randint(50, 255, size=3))
            self.objects.append(
                {"id": i + 1, "x": x, "y": y, "w": w, "h": h, "vx": vx, "vy": vy, "color": color}
            )

    def generate(self):
        frames, records = [], []
        for f in range(1, self.n_frames + 1):
            frame = np.full((self.height, self.width, 3), 30, dtype=np.uint8)
            cv2.line(frame, (0, self.height // 2), (self.width, self.height // 2), (60, 60, 60), 2)
            for obj in self.objects:
                obj["x"] += obj["vx"]
                obj["y"] += obj["vy"]
                if obj["x"] <= 0 or obj["x"] + obj["w"] >= self.width:
                    obj["vx"] *= -1
                    obj["x"] = float(np.clip(obj["x"], 0, self.width - obj["w"]))
                if obj["y"] <= 0 or obj["y"] + obj["h"] >= self.height:
                    obj["vy"] *= -1
                    obj["y"] = float(np.clip(obj["y"], 0, self.height - obj["h"]))
                x, y, w, h = int(obj["x"]), int(obj["y"]), obj["w"], obj["h"]
                cv2.rectangle(frame, (x, y), (x + w, y + h), obj["color"], -1)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 1)
                records.append({"frame": f, "id": obj["id"], "x": x, "y": y, "w": w, "h": h, "class": 1})
            frames.append(frame)
        return frames, pd.DataFrame(records)


USE_SYNTHETIC = True  # set False once DATA_ROOT points at a real downloaded SportsMOT sequence
DATA_ROOT = "/content/data/sportsmot"

if USE_SYNTHETIC or not os.path.isdir(DATA_ROOT):
    print("Using synthetic sequence (no real SportsMOT data found at DATA_ROOT).")
    frames, gt_df = SyntheticMOTSequence(n_frames=150, n_objects=6, seed=SEED).generate()
else:
    # Real SportsMOT (MOTChallenge layout): DATA_ROOT/<seq>/img1/*.jpg + DATA_ROOT/<seq>/gt/gt.txt
    # gt.txt columns: frame, id, x, y, w, h, conf, class, visibility
    seq_name = sorted(os.listdir(DATA_ROOT))[0]
    img_dir = os.path.join(DATA_ROOT, seq_name, "img1")
    gt_path = os.path.join(DATA_ROOT, seq_name, "gt", "gt.txt")
    frame_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    frames = [cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in frame_paths]
    gt_df = pd.read_csv(
        gt_path, header=None,
        names=["frame", "id", "x", "y", "w", "h", "conf", "class", "vis"],
    )

print(f"Loaded {len(frames)} frames, {gt_df['id'].nunique()} unique object IDs")


In [ ]:
def draw_boxes(img, boxes_df, color=(0, 255, 0)):
    img = img.copy()
    for _, row in boxes_df.iterrows():
        x, y, w, h = int(row["x"]), int(row["y"]), int(row["w"]), int(row["h"])
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        cv2.putText(img, str(int(row["id"])), (x, max(y - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img


sample_frame_idx = 10
sample_boxes = gt_df[gt_df["frame"] == sample_frame_idx + 1]
plt.figure(figsize=(8, 5))
plt.imshow(draw_boxes(frames[sample_frame_idx], sample_boxes))
plt.title(f"Frame {sample_frame_idx + 1} — ground truth")
plt.axis("off")
plt.show()


### 1.2 Normalization, augmentation, `Dataset` / `DataLoader`

Faster R-CNN normalizes internally (ImageNet mean/std) once given a `[0, 1]` float tensor,
so preprocessing here is: convert to tensor, then augment (colour jitter + horizontal flip,
train split only).


In [ ]:
class MOTDetectionDataset(Dataset):
    def __init__(self, frames, gt_df, frame_indices, augment=False):
        self.frames = frames
        self.gt_df = gt_df
        self.frame_indices = frame_indices
        self.augment = augment
        self.color_jitter = torchvision.transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)

    def __len__(self):
        return len(self.frame_indices)

    def __getitem__(self, idx):
        frame_idx = self.frame_indices[idx]
        img = self.frames[frame_idx]
        boxes_df = self.gt_df[self.gt_df["frame"] == frame_idx + 1]
        boxes = boxes_df[["x", "y", "w", "h"]].values.astype(np.float32)
        boxes_xyxy = np.stack(
            [boxes[:, 0], boxes[:, 1], boxes[:, 0] + boxes[:, 2], boxes[:, 1] + boxes[:, 3]], axis=1
        ) if len(boxes) else np.zeros((0, 4), dtype=np.float32)
        labels = np.ones(len(boxes_xyxy), dtype=np.int64)  # single 'player' class

        img_t = torch.from_numpy(img.copy()).permute(2, 0, 1).float() / 255.0

        if self.augment:
            img_t = self.color_jitter(img_t)
            if random.random() < 0.5 and len(boxes_xyxy):
                img_t = torch.flip(img_t, dims=[2])
                w = img.shape[1]
                boxes_xyxy = np.stack(
                    [w - boxes_xyxy[:, 2], boxes_xyxy[:, 1], w - boxes_xyxy[:, 0], boxes_xyxy[:, 3]], axis=1
                )

        target = {
            "boxes": torch.tensor(boxes_xyxy, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))


n_frames_total = len(frames)
split = int(n_frames_total * 0.8)
train_idx = list(range(split))
test_idx = list(range(split, n_frames_total))

train_ds = MOTDetectionDataset(frames, gt_df, train_idx, augment=True)
test_ds = MOTDetectionDataset(frames, gt_df, test_idx, augment=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Train frames: {len(train_ds)}, Test frames: {len(test_ds)}")


In [ ]:
img_t, target = train_ds[0]
img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

plt.figure(figsize=(8, 5))
plt.imshow(img_np)
ax = plt.gca()
for box in target["boxes"]:
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
plt.title("Augmented training sample")
plt.axis("off")
plt.show()


## 2. Model Development

### 2.1 Faster R-CNN — fine-tuning


In [ ]:
NUM_CLASSES = 2  # background + player


def build_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


model = build_model().to(device)
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

NUM_EPOCHS = 5
train_losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    lr_scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch + 1}: avg loss = {avg_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Avg training loss")
plt.title("Faster R-CNN fine-tuning loss")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
DETECTION_SCORE_THRESH = 0.5


@torch.no_grad()
def detect(model, img_t):
    model.eval()
    output = model([img_t.to(device)])[0]
    keep = output["scores"] >= DETECTION_SCORE_THRESH
    boxes = output["boxes"][keep].cpu().numpy()
    scores = output["scores"][keep].cpu().numpy()
    return boxes, scores


img_t, target = test_ds[0]
boxes, scores = detect(model, img_t)

img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
plt.figure(figsize=(8, 5))
plt.imshow(img_np)
ax = plt.gca()
for box, score in zip(boxes, scores):
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=2))
    ax.text(x1, max(y1 - 4, 0), f"{score:.2f}", color="red", fontsize=8)
plt.title("Detector output on a test frame")
plt.axis("off")
plt.show()


### 2.2 Tracking — Kalman filter + SORT + temporal consistency + adaptive gating

- **Kalman filter**: constant-velocity motion model per track (standard SORT state
  `[cx, cy, s, r, vx, vy, vs]`).
- **Temporal consistency**: a track's ID is only reported once it has been matched for
  `min_hits` consecutive frames (or during the initial warm-up window) — this suppresses
  flickering identities from single spurious detections.
- **Adaptive tracking**: the IoU gate used for matching loosens as a track's estimated
  speed increases, since fast-moving objects have less reliable position predictions.


In [ ]:
def iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_w, inter_h = max(0.0, inter_x2 - inter_x1), max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


class KalmanBoxTracker:
    """Constant-velocity Kalman filter tracking a single object's bbox (cx, cy, s, r)."""

    count = 0

    def __init__(self, bbox):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([
            [1, 0, 0, 0, 1, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 0, 1],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 1],
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 0],
        ])
        self.kf.R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.0
        self.kf.P *= 10.0
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = self._to_z(bbox)

        KalmanBoxTracker.count += 1
        self.id = KalmanBoxTracker.count
        self.hits = 1
        self.hit_streak = 1
        self.time_since_update = 0

    @staticmethod
    def _to_z(bbox):
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        cx, cy = x1 + w / 2, y1 + h / 2
        s = w * h
        r = w / h if h != 0 else 0.0
        return np.array([cx, cy, s, r]).reshape((4, 1))

    @staticmethod
    def _to_bbox(x):
        cx, cy, s, r = x[:4].reshape(-1)
        w = np.sqrt(max(s * r, 0.0))
        h = s / w if w != 0 else 0.0
        return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        return self._to_bbox(self.kf.x)

    def update(self, bbox):
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.kf.update(self._to_z(bbox))

    def get_state(self):
        return self._to_bbox(self.kf.x)

    def speed(self):
        vx, vy = self.kf.x[4, 0], self.kf.x[5, 0]
        return float(np.hypot(vx, vy))


class SortTracker:
    def __init__(self, max_age=15, min_hits=3, base_iou_threshold=0.3):
        self.max_age = max_age
        self.min_hits = min_hits
        self.base_iou_threshold = base_iou_threshold
        self.trackers = []
        self.frame_count = 0

    def _adaptive_threshold(self, trk):
        # faster-moving objects get a looser IoU gate: their predicted box is less reliable
        speed = trk.speed()
        return max(0.1, self.base_iou_threshold - min(0.15, speed * 0.01))

    def update(self, detections):
        self.frame_count += 1
        predicted_boxes = [trk.predict() for trk in self.trackers]

        matches, unmatched_dets, unmatched_trks = self._associate(detections, predicted_boxes)

        for t_idx, d_idx in matches:
            self.trackers[t_idx].update(detections[d_idx])

        for d_idx in unmatched_dets:
            self.trackers.append(KalmanBoxTracker(detections[d_idx]))

        results, alive = [], []
        for trk in self.trackers:
            if trk.time_since_update < self.max_age:
                alive.append(trk)
                # temporal consistency: confirm only after enough consecutive hits
                # (or during the initial warm-up window, per the standard SORT rule)
                if trk.time_since_update < 1 and (
                    trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits
                ):
                    results.append((trk.id, trk.get_state()))
        self.trackers = alive
        return results

    def _associate(self, detections, predicted_boxes):
        if len(self.trackers) == 0 or len(detections) == 0:
            return [], list(range(len(detections))), list(range(len(self.trackers)))

        iou_matrix = np.zeros((len(self.trackers), len(detections)), dtype=np.float32)
        for t, pbox in enumerate(predicted_boxes):
            for d, dbox in enumerate(detections):
                iou_matrix[t, d] = iou(pbox, dbox)

        row_idx, col_idx = linear_sum_assignment(-iou_matrix)

        matches, unmatched_trks, unmatched_dets = [], [], []
        matched_rows, matched_cols = set(row_idx), set(col_idx)
        for t in range(len(self.trackers)):
            if t not in matched_rows:
                unmatched_trks.append(t)
        for d in range(len(detections)):
            if d not in matched_cols:
                unmatched_dets.append(d)

        for t, d in zip(row_idx, col_idx):
            if iou_matrix[t, d] < self._adaptive_threshold(self.trackers[t]):
                unmatched_trks.append(t)
                unmatched_dets.append(d)
            else:
                matches.append((t, d))

        return matches, unmatched_dets, unmatched_trks


In [ ]:
def id_to_color(track_id):
    rng = np.random.RandomState(track_id)
    return tuple(int(c) for c in rng.randint(60, 255, size=3))


KalmanBoxTracker.count = 0
tracker = SortTracker(max_age=15, min_hits=3, base_iou_threshold=0.3)

track_results = []
annotated_frames = []

for frame_idx in tqdm(range(len(frames)), desc="Tracking"):
    img = frames[frame_idx]
    img_t = torch.from_numpy(img.copy()).permute(2, 0, 1).float() / 255.0
    boxes, _ = detect(model, img_t)

    tracked = tracker.update(boxes)

    vis = img.copy()
    for tid, box in tracked:
        x1, y1, x2, y2 = box.astype(int)
        color = id_to_color(tid)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis, f"ID {tid}", (x1, max(y1 - 6, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        track_results.append({"frame": frame_idx + 1, "id": tid, "x": x1, "y": y1, "w": x2 - x1, "h": y2 - y1})
    annotated_frames.append(vis)

track_df = pd.DataFrame(track_results)
print(f"Tracked {track_df['id'].nunique() if len(track_df) else 0} unique IDs across {len(frames)} frames")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_idxs = np.linspace(0, len(annotated_frames) - 1, 4).astype(int)
for ax, idx in zip(axes, sample_idxs):
    ax.imshow(annotated_frames[idx])
    ax.set_title(f"Frame {idx + 1}")
    ax.axis("off")
plt.suptitle("Tracked IDs across the sequence")
plt.show()


In [ ]:
out_path = "/content/tracked_output.mp4"
h, w = annotated_frames[0].shape[:2]
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 20, (w, h))
for f in annotated_frames:
    writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
writer.release()
print(f"Saved annotated video to {out_path}")


## 3. Evaluation

### 3.1 Detection — mAP


In [ ]:
metric = MeanAveragePrecision(iou_type="bbox")
model.eval()
preds, gts = [], []
with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for out, tgt in zip(outputs, targets):
            preds.append({"boxes": out["boxes"].cpu(), "scores": out["scores"].cpu(), "labels": out["labels"].cpu()})
            gts.append({"boxes": tgt["boxes"], "labels": tgt["labels"]})

metric.update(preds, gts)
map_result = metric.compute()
print(f"mAP@[.5:.95]: {map_result['map'].item():.4f}")
print(f"mAP@0.5:      {map_result['map_50'].item():.4f}")


### 3.2 Tracking — MOTA / MOTP / IDF1 / ID switches (via `py-motmetrics`)


In [ ]:
def build_accumulator(gt_df, trk_df, frame_numbers):
    acc = mm.MOTAccumulator(auto_id=True)
    empty_cols = ["frame", "id", "x", "y", "w", "h"]
    for f in frame_numbers:
        gt_frame = gt_df[gt_df["frame"] == f]
        trk_frame = trk_df[trk_df["frame"] == f] if len(trk_df) else pd.DataFrame(columns=empty_cols)

        gt_boxes = gt_frame[["x", "y", "w", "h"]].values
        trk_boxes = trk_frame[["x", "y", "w", "h"]].values if len(trk_frame) else np.empty((0, 4))

        dist_matrix = mm.distances.iou_matrix(gt_boxes, trk_boxes, max_iou=0.5)
        acc.update(gt_frame["id"].tolist(), trk_frame["id"].tolist(), dist_matrix)
    return acc


frame_numbers = sorted(gt_df["frame"].unique())
acc_sort = build_accumulator(gt_df, track_df, frame_numbers)

mh = mm.metrics.create()
summary = mh.compute(acc_sort, metrics=["mota", "motp", "idf1", "num_switches",
                                         "num_false_positives", "num_misses"], name="SORT+Kalman")
print(summary)


### 3.3 Inference speed (FPS)


In [ ]:
model.eval()
n_timing_frames = min(30, len(frames))
times = []
with torch.no_grad():
    for i in range(n_timing_frames):
        img_t = torch.from_numpy(frames[i].copy()).permute(2, 0, 1).float().to(device) / 255.0
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        _ = model([img_t])
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.time() - t0)

avg_time = float(np.mean(times))
print(f"Avg detector time/frame: {avg_time * 1000:.1f} ms  ->  {1 / avg_time:.1f} FPS (detector only, excludes tracker overhead)")


### 3.4 Optional baseline comparison — naive IoU tracker (no Kalman, no temporal consistency)

Shows what the Kalman filter + temporal consistency + adaptive gating actually buy you,
versus a bare-bones greedy IoU tracker matched only against the previous frame.


In [ ]:
class NaiveIouTracker:
    """Baseline: greedy IoU match to the previous frame only — no motion prediction,
    no ID-confirmation delay."""

    def __init__(self, iou_threshold=0.3):
        self.iou_threshold = iou_threshold
        self.prev_boxes = []
        self.prev_ids = []
        self.next_id = 1

    def update(self, detections):
        assigned_ids = [None] * len(detections)
        used_prev = set()
        if len(self.prev_boxes):
            iou_mat = np.zeros((len(detections), len(self.prev_boxes)))
            for i, db in enumerate(detections):
                for j, pb in enumerate(self.prev_boxes):
                    iou_mat[i, j] = iou(db, pb)
            for i in range(len(detections)):
                j = int(np.argmax(iou_mat[i])) if iou_mat.shape[1] else -1
                if j >= 0 and iou_mat[i, j] >= self.iou_threshold and j not in used_prev:
                    assigned_ids[i] = self.prev_ids[j]
                    used_prev.add(j)
        for i in range(len(detections)):
            if assigned_ids[i] is None:
                assigned_ids[i] = self.next_id
                self.next_id += 1
        self.prev_boxes = list(detections)
        self.prev_ids = assigned_ids
        return list(zip(assigned_ids, detections))


naive_tracker = NaiveIouTracker(iou_threshold=0.3)
baseline_results = []
for frame_idx in tqdm(range(len(frames)), desc="Baseline tracking"):
    img_t = torch.from_numpy(frames[frame_idx].copy()).permute(2, 0, 1).float() / 255.0
    boxes, _ = detect(model, img_t)
    for tid, box in naive_tracker.update(boxes):
        x1, y1, x2, y2 = box.astype(int)
        baseline_results.append({"frame": frame_idx + 1, "id": tid, "x": x1, "y": y1, "w": x2 - x1, "h": y2 - y1})

baseline_df = pd.DataFrame(baseline_results)
acc_baseline = build_accumulator(gt_df, baseline_df, frame_numbers)

summary_compare = mh.compute_many(
    [acc_sort, acc_baseline],
    metrics=["mota", "motp", "idf1", "num_switches"],
    names=["SORT + Kalman (ours)", "Naive IoU baseline"],
)
print(summary_compare)


## 4. Analysis & Justification

_Replace this section with observations drawn from the printed metrics/plots above once
you've run the notebook on your actual data (synthetic or real SportsMOT)._

- **Detection quality**: comment on the mAP achieved and the training loss curve — does
  loss plateau (good fit) or stay high/noisy (underfitting, needs more epochs/data)?
- **Tracking quality**: compare `SORT + Kalman` vs the naive baseline's MOTA/IDF1/switch
  count — quantify how much the Kalman motion model + temporal consistency reduced ID
  switches, especially around occlusion/overlap events.
- **Adaptive gating**: note whether fast-moving objects (high `speed()`) were tracked more
  reliably with the loosened IoU threshold than they would be with a fixed threshold.
- **Failure modes**: describe specific frames where tracking broke (e.g. two objects
  crossing paths, an object leaving and re-entering frame) and why.
- **Over/underfitting**: if using real data, discuss whether train/test loss or mAP suggest
  overfitting (small dataset, complex model) and what you'd do about it (more epochs vs.
  more data vs. stronger augmentation vs. freezing backbone layers).


## 5. Conclusion

_Summarize: problem tackled, approach (Faster R-CNN + SORT/Kalman + temporal consistency +
adaptive gating), key quantitative results, and main takeaway._

**Video presentation link:** _<add the 4-minute video link here, and also log it in the
shared Google Sheet as required by the assignment instructions>_
